In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql import types as T
import pandas as pd
import json
from functools import reduce

CATALOG = "toxictide"
SILVER = f"{CATALOG}.silver"
GOLD = f"{CATALOG}.gold"

#spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")

print("Notebook 05 setup complete")

Notebook 05 setup complete


In [0]:
def table_exists(table_name: str) -> bool:
    try:
        spark.read.table(table_name).limit(1).collect()
        return True
    except Exception:
        return False


def haversine_km_expr(lat1_col, lon1_col, lat2_col, lon2_col):
    return (
        F.lit(6371.0) * 2 * F.asin(
            F.sqrt(
                F.pow(F.sin((F.radians(lat2_col) - F.radians(lat1_col)) / 2), 2) +
                F.cos(F.radians(lat1_col)) * F.cos(F.radians(lat2_col)) *
                F.pow(F.sin((F.radians(lon2_col) - F.radians(lon1_col)) / 2), 2)
            )
        )
    )


def compute_tile_cols(df, lat_col="lat_dec", lon_col="lon_dec"):
    return (
        df.withColumn("lat_idx", F.floor(F.col(lat_col) * 111320 / 500))
          .withColumn(
              "lon_idx",
              F.floor(F.col(lon_col) * (111320 * F.cos(F.radians(F.col(lat_col)))) / 500)
          )
          .withColumn("tile_id", F.concat_ws("_", F.col("lat_idx"), F.col("lon_idx")))
    )


def get_numeric_feature_columns(df, exclude=None):
    exclude = set(exclude or [])
    numeric_types = {"double", "float", "int", "bigint", "smallint", "tinyint", "decimal", "long"}
    cols = []
    for field in df.schema.fields:
        dtype = field.dataType.simpleString().lower()
        if any(dtype.startswith(t) for t in numeric_types):
            if field.name not in exclude:
                cols.append(field.name)
    return cols


def prefix_feature_columns(df, prefix: str, feature_cols: list[str], keep_cols: list[str]):
    renamed = df
    for c in feature_cols:
        renamed = renamed.withColumnRenamed(c, f"{prefix}__{c}")
    return renamed.select(*(keep_cols + [f"{prefix}__{c}" for c in feature_cols]))


def save_table(df, table_name: str, mode: str = "overwrite"):
    df.write.format("delta").mode(mode).saveAsTable(table_name)

In [0]:
SITE_SEED_ROWS = [
    # Aquaculture / shellfish watch zones
    {
        "site_id": "aq_san_diego_bay",
        "site_name": "San Diego Bay Shellfish Watch",
        "site_type": "aquaculture",
        "region_name": "San Diego",
        "lat_dec": 32.66,
        "lon_dec": -117.13,
        "site_priority": 1,
        "notes": "Nearshore shellfish / embayment risk watch zone",
    },
    {
        "site_id": "aq_mission_bay",
        "site_name": "Mission Bay Shellfish Watch",
        "site_type": "aquaculture",
        "region_name": "San Diego",
        "lat_dec": 32.79,
        "lon_dec": -117.25,
        "site_priority": 1,
        "notes": "Semi-enclosed recreational/aquaculture-sensitive bay watch zone",
    },
    {
        "site_id": "aq_agua_hedionda",
        "site_name": "Agua Hedionda Lagoon Watch",
        "site_type": "aquaculture",
        "region_name": "North San Diego County",
        "lat_dec": 33.14,
        "lon_dec": -117.32,
        "site_priority": 1,
        "notes": "Lagoon / aquaculture-sensitive watch zone",
    },
    {
        "site_id": "aq_newport_bay",
        "site_name": "Newport Bay Shellfish Watch",
        "site_type": "aquaculture",
        "region_name": "Orange County",
        "lat_dec": 33.62,
        "lon_dec": -117.90,
        "site_priority": 2,
        "notes": "Embayment aquaculture watch zone",
    },
    {
        "site_id": "aq_morro_bay",
        "site_name": "Morro Bay Shellfish Watch",
        "site_type": "aquaculture",
        "region_name": "Central Coast",
        "lat_dec": 35.37,
        "lon_dec": -120.86,
        "site_priority": 2,
        "notes": "Central Coast shellfish/aquaculture watch zone",
    },
    {
        "site_id": "aq_humboldt_bay",
        "site_name": "Humboldt Bay Shellfish Watch",
        "site_type": "aquaculture",
        "region_name": "North Coast",
        "lat_dec": 40.77,
        "lon_dec": -124.20,
        "site_priority": 2,
        "notes": "North Coast shellfish/aquaculture watch zone",
    },

    # Beach companion shell
    {
        "site_id": "beach_la_jolla_shores",
        "site_name": "La Jolla Shores",
        "site_type": "beach",
        "region_name": "San Diego",
        "lat_dec": 32.85,
        "lon_dec": -117.26,
        "site_priority": 1,
        "notes": "Public-facing beach safety demo site",
    },
    {
        "site_id": "beach_mission_beach",
        "site_name": "Mission Beach",
        "site_type": "beach",
        "region_name": "San Diego",
        "lat_dec": 32.77,
        "lon_dec": -117.25,
        "site_priority": 1,
        "notes": "Public-facing beach safety demo site",
    },
    {
        "site_id": "beach_coronado",
        "site_name": "Coronado Central Beach",
        "site_type": "beach",
        "region_name": "San Diego",
        "lat_dec": 32.68,
        "lon_dec": -117.18,
        "site_priority": 1,
        "notes": "Public-facing beach safety demo site",
    },
]

In [0]:
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")
print("Arrow disabled")

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
import time

t0 = time.time()
print("about to run spark.range...")
x = spark.range(1).count()
print("spark.range(1).count() =", x, "elapsed =", round(time.time() - t0, 2), "s")

about to run spark.range...
spark.range(1).count() = 1 elapsed = 14.51 s


In [0]:
import time

t0 = time.time()
site_registry_pdf = pd.DataFrame(SITE_SEED_ROWS)
print(f"[1] built pandas df in {time.time() - t0:.2f}s, rows={len(site_registry_pdf)}")

t1 = time.time()
site_registry = spark.createDataFrame(site_registry_pdf)
print(f"[2] created spark df in {time.time() - t1:.2f}s")

t2 = time.time()
site_registry = (
    compute_tile_cols(site_registry, lat_col="lat_dec", lon_col="lon_dec")
    .withColumn(
        "default_match_radius_km",
        F.when(F.col("site_type") == "aquaculture", F.lit(25.0)).otherwise(F.lit(10.0))
    )
)
print(f"[3] added computed cols in {time.time() - t2:.2f}s")

t3 = time.time()
site_registry_count = site_registry.count()
print(f"[4] count() completed in {time.time() - t3:.2f}s, count={site_registry_count}")

t4 = time.time()
save_table(site_registry, f"{GOLD}.site_registry")
print(f"[5] save_table() completed in {time.time() - t4:.2f}s")

t5 = time.time()
display(site_registry)
print(f"[6] display() triggered in {time.time() - t5:.2f}s")

print(f"[total] {time.time() - t0:.2f}s")

[1] built pandas df in 0.00s, rows=9
[2] created spark df in 0.06s
[3] added computed cols in 0.05s
[4] count() completed in 0.86s, count=9
[5] save_table() completed in 21.16s


site_id,site_name,site_type,region_name,lat_dec,lon_dec,site_priority,notes,lat_idx,lon_idx,tile_id,default_match_radius_km
aq_san_diego_bay,San Diego Bay Shellfish Watch,aquaculture,San Diego,32.66,-117.13,1,Nearshore shellfish / embayment risk watch zone,7271,-21955,7271_-21955,25.0
aq_mission_bay,Mission Bay Shellfish Watch,aquaculture,San Diego,32.79,-117.25,1,Semi-enclosed recreational/aquaculture-sensitive bay watch zone,7300,-21946,7300_-21946,25.0
aq_agua_hedionda,Agua Hedionda Lagoon Watch,aquaculture,North San Diego County,33.14,-117.32,1,Lagoon / aquaculture-sensitive watch zone,7378,-21872,7378_-21872,25.0
aq_newport_bay,Newport Bay Shellfish Watch,aquaculture,Orange County,33.62,-117.9,2,Embayment aquaculture watch zone,7485,-21859,7485_-21859,25.0
aq_morro_bay,Morro Bay Shellfish Watch,aquaculture,Central Coast,35.37,-120.86,2,Central Coast shellfish/aquaculture watch zone,7874,-21942,7874_-21942,25.0
aq_humboldt_bay,Humboldt Bay Shellfish Watch,aquaculture,North Coast,40.77,-124.2,2,North Coast shellfish/aquaculture watch zone,9077,-20942,9077_-20942,25.0
beach_la_jolla_shores,La Jolla Shores,beach,San Diego,32.85,-117.26,1,Public-facing beach safety demo site,7313,-21933,7313_-21933,10.0
beach_mission_beach,Mission Beach,beach,San Diego,32.77,-117.25,1,Public-facing beach safety demo site,7295,-21951,7295_-21951,10.0
beach_coronado,Coronado Central Beach,beach,San Diego,32.68,-117.18,1,Public-facing beach safety demo site,7275,-21960,7275_-21960,10.0


[6] display() triggered in 1.52s
[total] 23.65s


In [0]:
SOURCE_CONFIGS = [
    {
        "source_name": "calcofi",
        "table_name": f"{SILVER}.calcofi_feature_mart",
        "prefix": "calcofi",
        "mode": "static",
        "radius_km": 50.0,
        "lookback_days": None,
        "feature_cols": [
            "depthm", "t_degc", "salnty", "o2ml_l", "stheta",
            "o2sat", "chlora", "po4um", "sio3um", "no2um", "no3um"
        ],
        "site_types": ["aquaculture", "beach"],
    },
    {
        "source_name": "ca_beach",
        "table_name": f"{SILVER}.ca_beach_feature_mart",
        "prefix": "ca_beach",
        "mode": "dynamic",
        "radius_km": 10.0,
        "lookback_days": 7,
        "feature_cols": ["enterococcus", "total_coliform", "fecal_coliform"],
        "site_types": ["beach", "aquaculture"],
    },
    {
        "source_name": "cce",
        "table_name": f"{SILVER}.cce_feature_mart",
        "prefix": "cce",
        "mode": "dynamic",
        "radius_km": 25.0,
        "lookback_days": 7,
        # omit oxygen for now because notebook 4 showed a weird table-display issue there
        "feature_cols": ["temp", "psal", "depth", "cndc", "no3"],
        "site_types": ["aquaculture", "beach"],
    },
    {
        "source_name": "cdip",
        "table_name": f"{SILVER}.cdip_feature_mart",
        "prefix": "cdip",
        "mode": "dynamic",
        "radius_km": 25.0,
        "lookback_days": 5,
        "feature_cols": ["wavehs", "wavetp", "sst"],
        "site_types": ["aquaculture", "beach"],
    },
    {
        "source_name": "noaa_tides",
        "table_name": f"{SILVER}.noaa_tides_feature_mart",
        "prefix": "tides",
        "mode": "dynamic",
        "radius_km": 50.0,
        "lookback_days": 3,
        "feature_cols": None,   # auto-detect numeric cols
        "site_types": ["aquaculture", "beach"],
    },
    {
        "source_name": "nws",
        "table_name": f"{SILVER}.nws_feature_mart",
        "prefix": "nws",
        "mode": "dynamic",
        "radius_km": 25.0,
        "lookback_days": 2,
        "feature_cols": None,
        "site_types": ["aquaculture", "beach"],
    },
    # future:
    # {
    #     "source_name": "argo",
    #     "table_name": f"{SILVER}.argo_feature_mart",
    #     "prefix": "argo",
    #     "mode": "dynamic",
    #     "radius_km": 75.0,
    #     "lookback_days": 14,
    #     "feature_cols": None,
    #     "site_types": ["aquaculture"],
    # },
]

In [0]:
available_source_tables = []

for cfg in SOURCE_CONFIGS:
    exists = table_exists(cfg["table_name"])
    available_source_tables.append({
        "source_name": cfg["source_name"],
        "table_name": cfg["table_name"],
        "exists": exists,
    })

available_source_tables_df = spark.createDataFrame(pd.DataFrame(available_source_tables))
display(available_source_tables_df)

source_name,table_name,exists
calcofi,toxictide.silver.calcofi_feature_mart,true
ca_beach,toxictide.silver.ca_beach_feature_mart,true
cce,toxictide.silver.cce_feature_mart,false
cdip,toxictide.silver.cdip_feature_mart,false
noaa_tides,toxictide.silver.noaa_tides_feature_mart,true
nws,toxictide.silver.nws_feature_mart,false


In [0]:
def align_source_to_sites(source_df, site_df, radius_km: float, allowed_site_types: list[str]):
    src = source_df.alias("src")
    sites = site_df.filter(F.col("site_type").isin(allowed_site_types)).alias("site_registry")

    joined = (
        src.crossJoin(F.broadcast(sites))
        .filter(F.abs(F.col("src.lat_dec") - F.col("site_registry.lat_dec")) <= F.lit(radius_km / 111.0))
        .filter(F.abs(F.col("src.lon_dec") - F.col("site_registry.lon_dec")) <= F.lit((radius_km / 111.0) * 1.5))
        .withColumn(
            "distance_km",
            haversine_km_expr(
                F.col("site_registry.lat_dec"),
                F.col("site_registry.lon_dec"),
                F.col("src.lat_dec"),
                F.col("src.lon_dec"),
            )
        )
        .filter(F.col("distance_km") <= F.lit(radius_km))
    )

    return joined

In [0]:
site_registry_alias = site_registry.alias("site_registry")

static_feature_parts = []

for cfg in SOURCE_CONFIGS:
    if cfg["mode"] != "static":
        continue
    if not table_exists(cfg["table_name"]):
        continue

    source_df = spark.read.table(cfg["table_name"])

    feature_cols = cfg["feature_cols"]
    if feature_cols is None:
        feature_cols = get_numeric_feature_columns(
            source_df,
            exclude={"lat_dec", "lon_dec", "tile_id", "conventional_date", "station_id"}
        )

    aligned = align_source_to_sites(
        source_df=source_df,
        site_df=site_registry_alias,
        radius_km=cfg["radius_km"],
        allowed_site_types=cfg["site_types"],
    )

    agg_exprs = [F.avg(c).alias(f"{cfg['prefix']}__{c}") for c in feature_cols]
    agg_exprs += [
        F.count("*").alias(f"{cfg['prefix']}__obs_count"),
        F.avg("distance_km").alias(f"{cfg['prefix']}__avg_distance_km"),
    ]

    static_site_features = (
        aligned
        .groupBy(
            F.col("site_registry.site_id").alias("site_id")
        )
        .agg(*agg_exprs)
    )

    static_feature_parts.append(static_site_features)

if static_feature_parts:
    static_features = reduce(
        lambda a, b: a.join(b, on="site_id", how="outer"),
        static_feature_parts
    )
else:
    static_features = spark.createDataFrame([], "site_id string")
    
display(static_features)

site_id,calcofi__depthm,calcofi__t_degc,calcofi__salnty,calcofi__o2ml_l,calcofi__stheta,calcofi__o2sat,calcofi__chlora,calcofi__po4um,calcofi__sio3um,calcofi__no2um,calcofi__no3um,calcofi__obs_count,calcofi__avg_distance_km
aq_newport_bay,135.15554305711785,11.675799798666166,33.7572496924063,3.621937481236866,25.640968717387093,61.19112847621236,0.7435410777385149,1.3741107058571118,21.224915686709576,0.05613687821612369,15.096145466923831,16002,24.48047624800918
beach_la_jolla_shores,151.49110663278717,11.340435845032879,33.79227749762258,3.4083293128074885,25.7388411849356,57.33573782366607,0.6130162539543995,1.5141380856310238,23.80890584857185,0.05450200153964581,17.055057101249268,22826,21.718718813246472
beach_mission_beach,154.37215691573212,11.266213194952455,33.797431812587405,3.3702829320492222,25.75786556362788,56.58577567538001,0.5708623873873848,1.5382879169529031,24.240960469425566,0.05407110272140942,17.401540148844433,22203,24.236571782185482
beach_coronado,152.2521493761298,11.313547613712753,33.79324794823722,3.400117640874683,25.745620761309343,57.10561886325745,0.5720870048954135,1.5294968601044097,24.085468606431856,0.054532188841201666,17.27774704785574,22681,34.16003858296314
aq_mission_bay,154.37215691573212,11.266213194952455,33.797431812587405,3.3702829320492227,25.757865563627888,56.58577567538002,0.5708623873873847,1.5382879169529025,24.24096046942558,0.054071102721409435,17.401540148844433,22203,23.400249922129596
aq_morro_bay,72.07146080938769,11.275814585535246,33.67163030869974,4.181590303232254,25.67359089648653,68.58106364101687,1.6250951974386323,1.3381119287374126,17.996187964048456,0.14298889786031496,14.845058397100264,6647,37.51092250695955
aq_agua_hedionda,148.77348040143912,11.39268230665899,33.788029613114105,3.4056790803829857,25.727447712014147,57.383316127565145,0.597718656468238,1.5166661618838484,23.859008670298476,0.053519840649898405,17.07202812961373,21124,32.33686045829122
aq_san_diego_bay,151.73591316242334,11.319984078575597,33.79237638460482,3.405636478517268,25.743895960246657,57.20276832749791,0.5720870048954135,1.5300818677986643,24.096273110855403,0.054532188841201666,17.277747047855744,22663,38.6996075148814
aq_humboldt_bay,188.25757575757575,8.432442748091603,33.60414225941422,3.334141414141414,26.101607594936706,51.41428571428572,1.08,2.27609756097561,23.8,0.17,16.5,264,35.98608862928815


In [0]:
dynamic_site_obs = {}

for cfg in SOURCE_CONFIGS:
    if cfg["mode"] != "dynamic":
        continue
    if not table_exists(cfg["table_name"]):
        print("Skipping missing source:", cfg["source_name"])
        continue

    print("Processing dynamic source:", cfg["source_name"])
    source_df = spark.read.table(cfg["table_name"])

    feature_cols = cfg["feature_cols"]
    if feature_cols is None:
        feature_cols = get_numeric_feature_columns(
            source_df,
            exclude={"lat_dec", "lon_dec", "tile_id", "conventional_date", "station_id", "lat_idx", "lon_idx"}
        )

    # keep only needed columns
    keep_cols = ["conventional_date", "lat_dec", "lon_dec"] + [c for c in feature_cols if c in source_df.columns]
    if "station_id" in source_df.columns:
        keep_cols = ["station_id"] + keep_cols

    source_df = source_df.select(*keep_cols)

    aligned = align_source_to_sites(
        source_df=source_df,
        site_df=site_registry_alias,
        radius_km=cfg["radius_km"],
        allowed_site_types=cfg["site_types"],
    )

    agg_exprs = [F.avg(c).alias(f"{cfg['prefix']}__{c}") for c in feature_cols if c in source_df.columns]
    agg_exprs += [
        F.min("distance_km").alias(f"{cfg['prefix']}__min_distance_km"),
        F.count("*").alias(f"{cfg['prefix']}__obs_count"),
    ]

    site_daily_obs = (
        aligned
        .groupBy(
            F.col("site_registry.site_id").alias("site_id"),
            F.col("conventional_date").alias("source_obs_date"),
        )
        .agg(*agg_exprs)
        .withColumnRenamed("source_obs_date", f"{cfg['prefix']}__source_obs_date")
    )

    dynamic_site_obs[cfg["prefix"]] = {
        "cfg": cfg,
        "df": site_daily_obs,
    }

    display(site_daily_obs.limit(10))

Processing dynamic source: ca_beach


site_id,ca_beach__source_obs_date,ca_beach__enterococcus,ca_beach__total_coliform,ca_beach__fecal_coliform,ca_beach__min_distance_km,ca_beach__obs_count
beach_la_jolla_shores,2020-01-08,8.833333333333334,null,null,1.5730924157924893,6
aq_mission_bay,2020-03-02,25.0,null,null,0.5366768263873407,8
beach_mission_beach,2019-03-20,134.5,null,null,1.2921717269629311,2
aq_mission_bay,2014-05-20,19.823529411764707,null,null,0.5366768263873407,17
beach_mission_beach,2015-07-07,40.294117647058826,null,null,0.24026739933991184,17
aq_mission_bay,2015-08-25,67.33333333333333,null,null,0.5366768263873407,18
aq_newport_bay,2012-12-05,16.22222222222222,null,null,2.1399315285196403,18
aq_newport_bay,2013-01-09,43.55555555555556,null,null,2.1399315285196403,18
aq_newport_bay,2013-07-02,33.76470588235294,null,null,2.1399315285196403,17
aq_agua_hedionda,2019-12-10,6.9,null,null,1.8132191683689545,10


Skipping missing source: cce
Skipping missing source: cdip
Processing dynamic source: noaa_tides


site_id,tides__source_obs_date,tides__water_level,tides__sigma,tides__i,tides__l,tides__min_distance_km,tides__obs_count
beach_coronado,2023-12-06,1.0255833333333333,0.009708333333333336,0.0,0.0,6.461637912994226,24
aq_agua_hedionda,2023-12-07,1.0239583333333335,0.01079166666666667,0.0,0.0,48.01343922640737,24
beach_la_jolla_shores,2023-12-10,1.029,0.0077500000000000025,0.0,0.0,15.288044207425077,24
aq_mission_bay,2023-12-13,1.1322916666666665,0.00829166666666667,0.0,0.0,8.551629921175172,24
aq_san_diego_bay,2023-12-15,1.0790833333333334,0.007791666666666669,0.0,0.0,11.595760346285378,24
aq_agua_hedionda,2023-12-16,1.0949166666666668,0.008291666666666668,0.0,0.0,48.01343922640737,24
beach_la_jolla_shores,2023-12-18,1.0792499999999998,0.009625000000000003,0.0,0.0,15.288044207425077,24
aq_san_diego_bay,2023-12-20,1.1072083333333336,0.0072916666666666685,0.0,0.0,11.595760346285378,24
beach_la_jolla_shores,2023-12-21,1.1228749999999998,0.006875000000000002,0.0,0.0,15.288044207425077,24
beach_la_jolla_shores,2023-12-28,1.0374999999999996,0.012208333333333335,0.0,0.0,15.288044207425077,24


Skipping missing source: nws


In [0]:
date_parts = []

for prefix, bundle in dynamic_site_obs.items():
    cfg = bundle["cfg"]
    df = bundle["df"].select(
        "site_id",
        F.col(f"{cfg['prefix']}__source_obs_date").alias("obs_date")
    )
    date_parts.append(df)

if date_parts:
    all_dates = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), date_parts).dropDuplicates()
    max_obs_date = all_dates.agg(F.max("obs_date").alias("max_date")).collect()[0]["max_date"]
else:
    max_obs_date = None

print("Max observed date:", max_obs_date)

if max_obs_date is None:
    # fallback: single current-date row per site
    base_calendar = site_registry.select("site_id").withColumn("calendar_date", F.current_date())
else:
    base_calendar = (
        site_registry.select("site_id")
        .withColumn("end_date", F.lit(max_obs_date))
        .withColumn("start_date", F.date_sub(F.col("end_date"), 29))
        .withColumn("calendar_date", F.explode(F.sequence(F.col("start_date"), F.col("end_date"))))
        .select("site_id", "calendar_date")
    )

display(base_calendar)

Max observed date: 2026-03-31


site_id,calendar_date
aq_san_diego_bay,2026-03-02
aq_san_diego_bay,2026-03-03
aq_san_diego_bay,2026-03-04
aq_san_diego_bay,2026-03-05
aq_san_diego_bay,2026-03-06
aq_san_diego_bay,2026-03-07
aq_san_diego_bay,2026-03-08
aq_san_diego_bay,2026-03-09
aq_san_diego_bay,2026-03-10
aq_san_diego_bay,2026-03-11


In [0]:
site_daily_features = base_calendar

for prefix, bundle in dynamic_site_obs.items():
    cfg = bundle["cfg"]
    obs_df = bundle["df"]

    lookback_days = cfg["lookback_days"]

    joined = (
        base_calendar.alias("cal")
        .join(
            obs_df.alias("obs"),
            on=F.col("cal.site_id") == F.col("obs.site_id"),
            how="left"
        )
        .filter(
            (F.col(f"obs.{prefix}__source_obs_date") <= F.col("cal.calendar_date")) &
            (F.col(f"obs.{prefix}__source_obs_date") >= F.date_sub(F.col("cal.calendar_date"), lookback_days))
        )
    )

    w = Window.partitionBy("cal.site_id", "cal.calendar_date").orderBy(F.col(f"obs.{prefix}__source_obs_date").desc())

    joined_latest = (
        joined
        .withColumn("rn", F.row_number().over(w))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )

    feature_cols = [c for c in obs_df.columns if c not in {"site_id", f"{prefix}__source_obs_date"}]

    select_cols = [
        F.col("cal.site_id").alias("site_id"),
        F.col("cal.calendar_date").alias("calendar_date"),
        F.col(f"obs.{prefix}__source_obs_date").alias(f"{prefix}__source_obs_date"),
        F.datediff(F.col("cal.calendar_date"), F.col(f"obs.{prefix}__source_obs_date")).alias(f"{prefix}__freshness_days"),
    ] + [F.col(f"obs.{c}").alias(c) for c in feature_cols]

    joined_latest = joined_latest.select(*select_cols)

    site_daily_features = site_daily_features.join(
        joined_latest,
        on=["site_id", "calendar_date"],
        how="left"
    )

display(site_daily_features)

site_id,calendar_date,ca_beach__source_obs_date,ca_beach__freshness_days,ca_beach__enterococcus,ca_beach__total_coliform,ca_beach__fecal_coliform,ca_beach__min_distance_km,ca_beach__obs_count,tides__source_obs_date,tides__freshness_days,tides__water_level,tides__sigma,tides__i,tides__l,tides__min_distance_km,tides__obs_count
aq_san_diego_bay,2026-03-02,2026-03-02,0,152061.0,null,null,2.831799698583906,6,2026-03-02,0,0.9561666666666665,0.010000000000000004,0.0,0.0,11.595760346285378,24
aq_san_diego_bay,2026-03-08,2026-03-04,4,39705.0,null,null,8.269098380645044,3,2026-03-08,0,0.9826250000000001,0.010000000000000004,0.041666666666666664,0.0,11.595760346285378,24
aq_san_diego_bay,2026-03-23,null,null,null,null,null,null,null,2026-03-23,0,1.0032916666666667,0.010000000000000004,0.0,0.0,11.595760346285378,24
aq_san_diego_bay,2026-03-29,null,null,null,null,null,null,null,2026-03-29,0,0.995,0.010000000000000004,0.0,0.0,11.595760346285378,24
aq_san_diego_bay,2026-03-22,null,null,null,null,null,null,null,2026-03-22,0,0.9973333333333332,0.010000000000000004,0.0,0.0,11.595760346285378,24
aq_san_diego_bay,2026-03-24,null,null,null,null,null,null,null,2026-03-24,0,0.9995416666666666,0.010000000000000004,0.0,0.0,11.595760346285378,24
aq_san_diego_bay,2026-03-30,null,null,null,null,null,null,null,2026-03-30,0,0.9797916666666668,0.010000000000000004,0.0,0.0,11.595760346285378,24
aq_san_diego_bay,2026-03-05,2026-03-04,1,39705.0,null,null,8.269098380645044,3,2026-03-05,0,0.952083333333333,0.010000000000000004,0.0,0.0,11.595760346285378,24
aq_san_diego_bay,2026-03-19,null,null,null,null,null,null,null,2026-03-19,0,0.9924583333333338,0.010000000000000004,0.0,0.0,11.595760346285378,24
aq_san_diego_bay,2026-03-31,null,null,null,null,null,null,null,2026-03-31,0,1.0028749999999997,0.010000000000000004,0.041666666666666664,0.0,11.595760346285378,24


In [0]:
site_daily_features = (
    site_daily_features
    .join(site_registry, on="site_id", how="left")
    .join(static_features, on="site_id", how="left")
)

display(site_daily_features)

site_id,calendar_date,ca_beach__source_obs_date,ca_beach__freshness_days,ca_beach__enterococcus,ca_beach__total_coliform,ca_beach__fecal_coliform,ca_beach__min_distance_km,ca_beach__obs_count,tides__source_obs_date,tides__freshness_days,tides__water_level,tides__sigma,tides__i,tides__l,tides__min_distance_km,tides__obs_count,site_name,site_type,region_name,lat_dec,lon_dec,site_priority,notes,lat_idx,lon_idx,tile_id,default_match_radius_km,calcofi__depthm,calcofi__t_degc,calcofi__salnty,calcofi__o2ml_l,calcofi__stheta,calcofi__o2sat,calcofi__chlora,calcofi__po4um,calcofi__sio3um,calcofi__no2um,calcofi__no3um,calcofi__obs_count,calcofi__avg_distance_km
aq_san_diego_bay,2026-03-23,null,null,null,null,null,null,null,2026-03-23,0,1.0032916666666667,0.010000000000000004,0.0,0.0,11.595760346285378,24,San Diego Bay Shellfish Watch,aquaculture,San Diego,32.66,-117.13,1,Nearshore shellfish / embayment risk watch zone,7271,-21955,7271_-21955,25.0,151.73591316242334,11.319984078575597,33.79237638460482,3.405636478517268,25.743895960246657,57.20276832749791,0.5720870048954135,1.5300818677986643,24.096273110855403,0.054532188841201666,17.277747047855744,22663,38.6996075148814
aq_san_diego_bay,2026-03-29,null,null,null,null,null,null,null,2026-03-29,0,0.995,0.010000000000000004,0.0,0.0,11.595760346285378,24,San Diego Bay Shellfish Watch,aquaculture,San Diego,32.66,-117.13,1,Nearshore shellfish / embayment risk watch zone,7271,-21955,7271_-21955,25.0,151.73591316242334,11.319984078575597,33.79237638460482,3.405636478517268,25.743895960246657,57.20276832749791,0.5720870048954135,1.5300818677986643,24.096273110855403,0.054532188841201666,17.277747047855744,22663,38.6996075148814
aq_san_diego_bay,2026-03-09,2026-03-04,5,39705.0,null,null,8.269098380645044,3,2026-03-09,0,1.0030416666666666,0.010000000000000004,0.0,0.0,11.595760346285378,24,San Diego Bay Shellfish Watch,aquaculture,San Diego,32.66,-117.13,1,Nearshore shellfish / embayment risk watch zone,7271,-21955,7271_-21955,25.0,151.73591316242334,11.319984078575597,33.79237638460482,3.405636478517268,25.743895960246657,57.20276832749791,0.5720870048954135,1.5300818677986643,24.096273110855403,0.054532188841201666,17.277747047855744,22663,38.6996075148814
aq_san_diego_bay,2026-03-02,2026-03-02,0,152061.0,null,null,2.831799698583906,6,2026-03-02,0,0.9561666666666665,0.010000000000000004,0.0,0.0,11.595760346285378,24,San Diego Bay Shellfish Watch,aquaculture,San Diego,32.66,-117.13,1,Nearshore shellfish / embayment risk watch zone,7271,-21955,7271_-21955,25.0,151.73591316242334,11.319984078575597,33.79237638460482,3.405636478517268,25.743895960246657,57.20276832749791,0.5720870048954135,1.5300818677986643,24.096273110855403,0.054532188841201666,17.277747047855744,22663,38.6996075148814
aq_san_diego_bay,2026-03-13,null,null,null,null,null,null,null,2026-03-13,0,0.9643333333333336,0.010000000000000004,0.041666666666666664,0.0,11.595760346285378,24,San Diego Bay Shellfish Watch,aquaculture,San Diego,32.66,-117.13,1,Nearshore shellfish / embayment risk watch zone,7271,-21955,7271_-21955,25.0,151.73591316242334,11.319984078575597,33.79237638460482,3.405636478517268,25.743895960246657,57.20276832749791,0.5720870048954135,1.5300818677986643,24.096273110855403,0.054532188841201666,17.277747047855744,22663,38.6996075148814
aq_san_diego_bay,2026-03-26,null,null,null,null,null,null,null,2026-03-26,0,0.9789166666666667,0.010000000000000004,0.0,0.0,11.595760346285378,24,San Diego Bay Shellfish Watch,aquaculture,San Diego,32.66,-117.13,1,Nearshore shellfish / embayment risk watch zone,7271,-21955,7271_-21955,25.0,151.73591316242334,11.319984078575597,33.79237638460482,3.405636478517268,25.743895960246657,57.20276832749791,0.5720870048954135,1.5300818677986643,24.096273110855403,0.054532188841201666,17.277747047855744,22663,38.6996075148814
aq_san_diego_bay,2026-03-07,2026-03-04,3,39705.0,null,null,8.269098380645044,3,2026-03-07,0,0.9425000000000002,0.010000000000000004,0.0,0.0,11.595760346285378,24,San Diego 

In [0]:
availability_cols = []

for cfg in SOURCE_CONFIGS:
    prefix = cfg["prefix"]
    source_obs_col = f"{prefix}__source_obs_date"
    if source_obs_col in site_daily_features.columns:
        avail_col = f"{prefix}__available"
        site_daily_features = site_daily_features.withColumn(
            avail_col,
            F.when(F.col(source_obs_col).isNotNull(), F.lit(1)).otherwise(F.lit(0))
        )
        availability_cols.append(avail_col)

site_daily_features = site_daily_features.withColumn(
    "n_sources_available",
    sum([F.col(c) for c in availability_cols]) if availability_cols else F.lit(0)
)

display(site_daily_features)

site_id,calendar_date,ca_beach__source_obs_date,ca_beach__freshness_days,ca_beach__enterococcus,ca_beach__total_coliform,ca_beach__fecal_coliform,ca_beach__min_distance_km,ca_beach__obs_count,tides__source_obs_date,tides__freshness_days,tides__water_level,tides__sigma,tides__i,tides__l,tides__min_distance_km,tides__obs_count,site_name,site_type,region_name,lat_dec,lon_dec,site_priority,notes,lat_idx,lon_idx,tile_id,default_match_radius_km,calcofi__depthm,calcofi__t_degc,calcofi__salnty,calcofi__o2ml_l,calcofi__stheta,calcofi__o2sat,calcofi__chlora,calcofi__po4um,calcofi__sio3um,calcofi__no2um,calcofi__no3um,calcofi__obs_count,calcofi__avg_distance_km,ca_beach__available,tides__available,n_sources_available
aq_san_diego_bay,2026-03-23,null,null,null,null,null,null,null,2026-03-23,0,1.0032916666666667,0.010000000000000004,0.0,0.0,11.595760346285378,24,San Diego Bay Shellfish Watch,aquaculture,San Diego,32.66,-117.13,1,Nearshore shellfish / embayment risk watch zone,7271,-21955,7271_-21955,25.0,151.73591316242334,11.319984078575597,33.79237638460482,3.405636478517268,25.743895960246657,57.20276832749791,0.5720870048954135,1.5300818677986643,24.096273110855403,0.054532188841201666,17.277747047855744,22663,38.6996075148814,0,1,1
aq_san_diego_bay,2026-03-29,null,null,null,null,null,null,null,2026-03-29,0,0.995,0.010000000000000004,0.0,0.0,11.595760346285378,24,San Diego Bay Shellfish Watch,aquaculture,San Diego,32.66,-117.13,1,Nearshore shellfish / embayment risk watch zone,7271,-21955,7271_-21955,25.0,151.73591316242334,11.319984078575597,33.79237638460482,3.405636478517268,25.743895960246657,57.20276832749791,0.5720870048954135,1.5300818677986643,24.096273110855403,0.054532188841201666,17.277747047855744,22663,38.6996075148814,0,1,1
aq_san_diego_bay,2026-03-09,2026-03-04,5,39705.0,null,null,8.269098380645044,3,2026-03-09,0,1.0030416666666666,0.010000000000000004,0.0,0.0,11.595760346285378,24,San Diego Bay Shellfish Watch,aquaculture,San Diego,32.66,-117.13,1,Nearshore shellfish / embayment risk watch zone,7271,-21955,7271_-21955,25.0,151.73591316242334,11.319984078575597,33.79237638460482,3.405636478517268,25.743895960246657,57.20276832749791,0.5720870048954135,1.5300818677986643,24.096273110855403,0.054532188841201666,17.277747047855744,22663,38.6996075148814,1,1,2
aq_san_diego_bay,2026-03-02,2026-03-02,0,152061.0,null,null,2.831799698583906,6,2026-03-02,0,0.9561666666666665,0.010000000000000004,0.0,0.0,11.595760346285378,24,San Diego Bay Shellfish Watch,aquaculture,San Diego,32.66,-117.13,1,Nearshore shellfish / embayment risk watch zone,7271,-21955,7271_-21955,25.0,151.73591316242334,11.319984078575597,33.79237638460482,3.405636478517268,25.743895960246657,57.20276832749791,0.5720870048954135,1.5300818677986643,24.096273110855403,0.054532188841201666,17.277747047855744,22663,38.6996075148814,1,1,2
aq_san_diego_bay,2026-03-13,null,null,null,null,null,null,null,2026-03-13,0,0.9643333333333336,0.010000000000000004,0.041666666666666664,0.0,11.595760346285378,24,San Diego Bay Shellfish Watch,aquaculture,San Diego,32.66,-117.13,1,Nearshore shellfish / embayment risk watch zone,7271,-21955,7271_-21955,25.0,151.73591316242334,11.319984078575597,33.79237638460482,3.405636478517268,25.743895960246657,57.20276832749791,0.5720870048954135,1.5300818677986643,24.096273110855403,0.054532188841201666,17.277747047855744,22663,38.6996075148814,0,1,1
aq_san_diego_bay,2026-03-26,null,null,null,null,null,null,null,2026-03-26,0,0.9789166666666667,0.010000000000000004,0.0,0.0,11.595760346285378,24,San Diego Bay Shellfish Watch,aquaculture,San Diego,32.66,-117.13,1,Nearshore shellfish / embayment risk watch zone,7271,-21955,7271_-21955,25.0,151.73591316242334,11.319984078575597,33.79237638460482,3.405636478517268,25.743895960246657,57.20276832749791,0.5720870048954135,1.5300818677986643,24.096273110855403,0.054532188841201666,17.277747047855744,22663,38.6996075148814,0,1,1
aq_san_diego_bay,2026-03-07,2026-03-04,3,39705.0,null,null,8.269098380645044,3,

In [0]:
save_table(site_daily_features, f"{GOLD}.site_daily_features")
display(spark.read.table(f"{GOLD}.site_daily_features"))

site_id,calendar_date,ca_beach__source_obs_date,ca_beach__freshness_days,ca_beach__enterococcus,ca_beach__total_coliform,ca_beach__fecal_coliform,ca_beach__min_distance_km,ca_beach__obs_count,tides__source_obs_date,tides__freshness_days,tides__water_level,tides__sigma,tides__i,tides__l,tides__min_distance_km,tides__obs_count,site_name,site_type,region_name,lat_dec,lon_dec,site_priority,notes,lat_idx,lon_idx,tile_id,default_match_radius_km,calcofi__depthm,calcofi__t_degc,calcofi__salnty,calcofi__o2ml_l,calcofi__stheta,calcofi__o2sat,calcofi__chlora,calcofi__po4um,calcofi__sio3um,calcofi__no2um,calcofi__no3um,calcofi__obs_count,calcofi__avg_distance_km,ca_beach__available,tides__available,n_sources_available
beach_mission_beach,2026-03-03,2026-02-27,4,20737.0,null,null,4.65112575514281,1,2026-03-03,0,0.9556666666666667,0.010000000000000004,0.041666666666666664,0.0,6.363094884599217,24,Mission Beach,beach,San Diego,32.77,-117.25,1,Public-facing beach safety demo site,7295,-21951,7295_-21951,10.0,154.37215691573212,11.266213194952455,33.797431812587405,3.3702829320492222,25.75786556362788,56.58577567538001,0.5708623873873848,1.5382879169529031,24.240960469425566,0.05407110272140942,17.401540148844433,22203,24.236571782185482,1,1,2
beach_mission_beach,2026-03-21,null,null,null,null,null,null,null,2026-03-21,0,0.9974166666666666,0.010000000000000004,0.0,0.0,6.363094884599217,24,Mission Beach,beach,San Diego,32.77,-117.25,1,Public-facing beach safety demo site,7295,-21951,7295_-21951,10.0,154.37215691573212,11.266213194952455,33.797431812587405,3.3702829320492222,25.75786556362788,56.58577567538001,0.5708623873873848,1.5382879169529031,24.240960469425566,0.05407110272140942,17.401540148844433,22203,24.236571782185482,0,1,1
beach_coronado,2026-03-06,2026-03-02,4,17767.0,null,null,0.5331158539306972,3,2026-03-06,0,0.9478333333333334,0.010000000000000004,0.0,0.0,6.461637912994226,24,Coronado Central Beach,beach,San Diego,32.68,-117.18,1,Public-facing beach safety demo site,7275,-21960,7275_-21960,10.0,152.2521493761298,11.313547613712753,33.79324794823722,3.400117640874683,25.745620761309343,57.10561886325745,0.5720870048954135,1.5294968601044097,24.085468606431856,0.054532188841201666,17.27774704785574,22681,34.16003858296314,1,1,2
beach_coronado,2026-03-13,null,null,null,null,null,null,null,2026-03-13,0,0.9643333333333333,0.010000000000000004,0.041666666666666664,0.0,6.461637912994226,24,Coronado Central Beach,beach,San Diego,32.68,-117.18,1,Public-facing beach safety demo site,7275,-21960,7275_-21960,10.0,152.2521493761298,11.313547613712753,33.79324794823722,3.400117640874683,25.745620761309343,57.10561886325745,0.5720870048954135,1.5294968601044097,24.085468606431856,0.054532188841201666,17.27774704785574,22681,34.16003858296314,0,1,1
beach_coronado,2026-03-18,null,null,null,null,null,null,null,2026-03-18,0,0.9956666666666668,0.010000000000000004,0.0,0.0,6.461637912994226,24,Coronado Central Beach,beach,San Diego,32.68,-117.18,1,Public-facing beach safety demo site,7275,-21960,7275_-21960,10.0,152.2521493761298,11.313547613712753,33.79324794823722,3.400117640874683,25.745620761309343,57.10561886325745,0.5720870048954135,1.5294968601044097,24.085468606431856,0.054532188841201666,17.27774704785574,22681,34.16003858296314,0,1,1
beach_coronado,2026-03-24,null,null,null,null,null,null,null,2026-03-24,0,0.9995416666666666,0.010000000000000004,0.0,0.0,6.461637912994226,24,Coronado Central Beach,beach,San Diego,32.68,-117.18,1,Public-facing beach safety demo site,7275,-21960,7275_-21960,10.0,152.2521493761298,11.313547613712753,33.79324794823722,3.400117640874683,25.745620761309343,57.10561886325745,0.5720870048954135,1.5294968601044097,24.085468606431856,0.054532188841201666,17.27774704785574,22681,34.16003858296314,0,1,1
beach_coronado,2026-03-29,null,null,null,null,null,null,null,2026-03-29,0,0.995,0.010000000000000004,0.0,0.0,6.461637912994226,24,Coronado Central Beach,beach,San Diego,32.68,-117.18,1,Public-facing beach safety demo sit

In [0]:
site_registry_df = spark.read.table(f"{GOLD}.site_registry")
site_daily_df = spark.read.table(f"{GOLD}.site_daily_features")

print("site_registry rows =", site_registry_df.count())
print("site_daily_features rows =", site_daily_df.count())
print("site_daily_features cols =", len(site_daily_df.columns))
print("site_daily_features columns =", site_daily_df.columns)

display(
    site_daily_df.groupBy("site_type")
    .agg(
        F.count("*").alias("n_rows"),
        F.max("calendar_date").alias("max_date"),
        F.avg("n_sources_available").alias("avg_sources_available"),
    )
)

site_registry rows = 9
site_daily_features rows = 270
site_daily_features cols = 44
site_daily_features columns = ['site_id', 'calendar_date', 'ca_beach__source_obs_date', 'ca_beach__freshness_days', 'ca_beach__enterococcus', 'ca_beach__total_coliform', 'ca_beach__fecal_coliform', 'ca_beach__min_distance_km', 'ca_beach__obs_count', 'tides__source_obs_date', 'tides__freshness_days', 'tides__water_level', 'tides__sigma', 'tides__i', 'tides__l', 'tides__min_distance_km', 'tides__obs_count', 'site_name', 'site_type', 'region_name', 'lat_dec', 'lon_dec', 'site_priority', 'notes', 'lat_idx', 'lon_idx', 'tile_id', 'default_match_radius_km', 'calcofi__depthm', 'calcofi__t_degc', 'calcofi__salnty', 'calcofi__o2ml_l', 'calcofi__stheta', 'calcofi__o2sat', 'calcofi__chlora', 'calcofi__po4um', 'calcofi__sio3um', 'calcofi__no2um', 'calcofi__no3um', 'calcofi__obs_count', 'calcofi__avg_distance_km', 'ca_beach__available', 'tides__available', 'n_sources_available']


site_type,n_rows,max_date,avg_sources_available
beach,90,2026-03-31,1.2555555555555555
aquaculture,180,2026-03-31,0.6555555555555556
